In [ ]:
%pip install -q torch torchvision scikit-learn matplotlib seaborn pandas

In [ ]:
import os
import time
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, models, transforms

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 2
LEARNING_RATE = 0.001

TRAIN_PER_CLASS = 20
VAL_PER_CLASS = 5
TEST_PER_CLASS = 5
NUM_CLASSES = 37

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
                               
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

                                                 
trainval_base = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    target_types="category",
    download=True,
    transform=None
)

test_base = datasets.OxfordIIITPet(
    root="./data",
    split="test",
    target_types="category",
    download=True,
    transform=None
)

print("Train + validation images:", len(trainval_base))
print("Test images available:", len(test_base))
print("Number of classes:", len(trainval_base.classes))


In [ ]:
def get_dataset_labels(dataset):
    """Read integer class labels from a torchvision dataset."""
    if hasattr(dataset, "_labels"):
        return list(dataset._labels)
    if hasattr(dataset, "targets"):
        return list(dataset.targets)
                                                                          
    return [dataset[i][1] for i in range(len(dataset))]


trainval_labels = get_dataset_labels(trainval_base)
test_labels = get_dataset_labels(test_base)

class_names = list(trainval_base.classes)
print("First 10 class names:", class_names[:10])
print("Trainval label count:", len(trainval_labels))
print("Test label count:", len(test_labels))


In [ ]:
def balanced_train_val_indices(labels, train_count, val_count, num_classes):
    grouped = {class_id: [] for class_id in range(num_classes)}

    for index, label in enumerate(labels):
        grouped[int(label)].append(index)

    train_indices = []
    val_indices = []

    for class_id in range(num_classes):
        class_indices = grouped[class_id].copy()
        random.shuffle(class_indices)

        required = train_count + val_count
        if len(class_indices) < required:
            raise ValueError(
                f"Class {class_id} has only {len(class_indices)} images; "
                f"{required} are required."
            )

        train_indices.extend(class_indices[:train_count])
        val_indices.extend(class_indices[train_count:required])

    return train_indices, val_indices


def balanced_test_indices(labels, count, num_classes):
    grouped = {class_id: [] for class_id in range(num_classes)}

    for index, label in enumerate(labels):
        grouped[int(label)].append(index)

    selected = []
    for class_id in range(num_classes):
        class_indices = grouped[class_id].copy()
        random.shuffle(class_indices)

        if len(class_indices) < count:
            raise ValueError(
                f"Test class {class_id} has only {len(class_indices)} images; "
                f"{count} are required."
            )

        selected.extend(class_indices[:count])

    return selected


train_indices, val_indices = balanced_train_val_indices(
    trainval_labels, TRAIN_PER_CLASS, VAL_PER_CLASS, NUM_CLASSES
)
test_indices = balanced_test_indices(test_labels, TEST_PER_CLASS, NUM_CLASSES)

print("Training samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Testing samples:", len(test_indices))


In [ ]:
class PetSubset(Dataset):
    """Apply a transform to selected examples from a base dataset."""
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        image, label = self.base_dataset[self.indices[item]]
        if self.transform is not None:
            image = self.transform(image)
        return image, int(label)


train_dataset = PetSubset(trainval_base, train_indices, train_transform)
val_dataset = PetSubset(trainval_base, val_indices, eval_transform)
test_dataset = PetSubset(test_base, test_indices, eval_transform)

                                                               
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print(f"Classes: {NUM_CLASSES}")
print(f"Train: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}")


In [ ]:
                                                                       
def show_batch(loader, class_names, count=8):
    images, labels = next(iter(loader))
    count = min(count, len(images))

    mean = torch.tensor(imagenet_mean).view(3, 1, 1)
    std = torch.tensor(imagenet_std).view(3, 1, 1)

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, image, label in zip(axes.flat, images[:count], labels[:count]):
        image = (image * std + mean).clamp(0, 1)
        ax.imshow(image.permute(1, 2, 0).numpy())
        ax.set_title(class_names[int(label)])
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_batch(train_loader, class_names)


In [ ]:
def build_resnet50(num_classes=NUM_CLASSES):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    for parameter in model.parameters():
        parameter.requires_grad = False

    input_features = model.fc.in_features
    model.fc = nn.Linear(input_features, num_classes)
    return model


def build_vit(num_classes=NUM_CLASSES):
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

    for parameter in model.parameters():
        parameter.requires_grad = False

    input_features = model.heads.head.in_features
    model.heads.head = nn.Linear(input_features, num_classes)
    return model


resnet_model = build_resnet50().to(DEVICE)
vit_model = build_vit().to(DEVICE)

print("ResNet50 total parameters:", sum(p.numel() for p in resnet_model.parameters()))
print("ResNet50 trainable parameters:", sum(p.numel() for p in resnet_model.parameters() if p.requires_grad))
print("ViT-B/16 total parameters:", sum(p.numel() for p in vit_model.parameters()))
print("ViT-B/16 trainable parameters:", sum(p.numel() for p in vit_model.parameters() if p.requires_grad))


In [ ]:
criterion = nn.CrossEntropyLoss()


def run_epoch(model, loader, training=False, optimizer=None):
    if training and optimizer is None:
        raise ValueError("An optimizer is required for training.")

    model.train() if training else model.eval()

    total_loss = 0.0
    all_predictions = []
    all_labels = []

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)

            all_predictions.extend(predictions.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    average_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_labels, all_predictions)

    return average_loss, accuracy, np.array(all_labels), np.array(all_predictions)


In [ ]:
def train_model(model_name, model):
    print("\n" + "=" * 60)
    print(f"MODEL: {model_name}")
    print("=" * 60)

    trainable_parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(trainable_parameters, lr=LEARNING_RATE)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    start_time = time.time()

    for epoch in range(EPOCHS):
        epoch_start = time.time()

        train_loss, train_acc, _, _ = run_epoch(
            model, train_loader, training=True, optimizer=optimizer
        )
        val_loss, val_acc, _, _ = run_epoch(
            model, val_loader, training=False
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc * 100:.2f}% | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc * 100:.2f}% | "
            f"Time: {(time.time() - epoch_start) / 60:.2f} min"
        )

    total_time = time.time() - start_time
    test_loss, _, y_true, y_pred = run_epoch(model, test_loader, training=False)

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Test Loss": test_loss,
        "Training Time (min)": total_time / 60,
        "History": history,
        "y_true": y_true,
        "y_pred": y_pred
    }

    print("\nTEST RESULTS")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Accuracy: {results['Accuracy'] * 100:.2f}%")
    print(f"Macro Precision: {results['Macro Precision'] * 100:.2f}%")
    print(f"Macro Recall: {results['Macro Recall'] * 100:.2f}%")
    print(f"Macro F1: {results['Macro F1'] * 100:.2f}%")
    print(f"Total model time: {total_time / 60:.2f} minutes")

    return results


resnet_results = train_model("ResNet50 CNN", resnet_model)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

vit_results = train_model("ViT-B/16", vit_model)

results = [resnet_results, vit_results]


In [ ]:
metric_names = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]

comparison = pd.DataFrame([
    {
        "Model": result["Model"],
        "Accuracy (%)": result["Accuracy"] * 100,
        "Macro Precision (%)": result["Macro Precision"] * 100,
        "Macro Recall (%)": result["Macro Recall"] * 100,
        "Macro F1 (%)": result["Macro F1"] * 100,
        "Test Loss": result["Test Loss"],
        "Training Time (min)": result["Training Time (min)"]
    }
    for result in results
])

display(comparison.round(2))

plot_data = comparison.set_index("Model")[
    ["Accuracy (%)", "Macro Precision (%)", "Macro Recall (%)", "Macro F1 (%)"]
]
ax = plot_data.plot(kind="bar", figsize=(11, 6), ylim=(0, 100))
ax.set_title("ViT vs ResNet50: Classification Metrics")
ax.set_ylabel("Score (%)")
ax.set_xlabel("Model")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.legend(loc="best")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, result in zip(axes, results):
    history = result["History"]
    epochs_range = range(1, len(history["train_acc"]) + 1)

    ax.plot(epochs_range, [x * 100 for x in history["train_acc"]],
            marker="o", label="Train")
    ax.plot(epochs_range, [x * 100 for x in history["val_acc"]],
            marker="o", label="Validation")

    ax.set_title(result["Model"])
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)")
    ax.legend()
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, result in zip(axes, results):
    cm = confusion_matrix(
        result["y_true"],
        result["y_pred"],
        labels=list(range(NUM_CLASSES))
    )
    sns.heatmap(
        cm,
        ax=ax,
        cmap="Blues",
        cbar=False,
        xticklabels=class_names,
        yticklabels=class_names
    )
    ax.set_title(f"{result['Model']} Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.tick_params(axis="x", labelrotation=90, labelsize=6)
    ax.tick_params(axis="y", labelrotation=0, labelsize=6)

plt.tight_layout()
plt.show()


In [ ]:
for result in results:
    print("\n" + "=" * 70)
    print("Classification report:", result["Model"])
    print("=" * 70)
    print(classification_report(
        result["y_true"],
        result["y_pred"],
        labels=list(range(NUM_CLASSES)),
        target_names=class_names,
        zero_division=0
    ))


In [ ]:
best_model = max(results, key=lambda item: item["Accuracy"])

print("FINAL COMPARISON")
for result in results:
    print(
        f"{result['Model']}: "
        f"Accuracy={result['Accuracy'] * 100:.2f}% | "
        f"Precision={result['Macro Precision'] * 100:.2f}% | "
        f"Recall={result['Macro Recall'] * 100:.2f}% | "
        f"F1={result['Macro F1'] * 100:.2f}% | "
        f"Time={result['Training Time (min)']:.2f} min"
    )

print(f"\nHigher test accuracy in this run: {best_model['Model']}")
